In [1]:
from comet import download_model, load_from_checkpoint
import pandas as pd
from tqdm import tqdm

In [2]:
# Choose your model from Hugging Face Hub
model_path = download_model("Unbabel/wmt22-comet-da")

# Load the model checkpoint:
model = load_from_checkpoint(model_path)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/Jordan/anaconda3/envs/responsible_ml/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [3]:
flores_output = pd.read_parquet('data/test_set/flores_mt_output.parquet')
flores_opus = (flores_output
               .loc[:,['text_eng', 'text_fra', 'mt_opus']]
               .rename(columns={'text_eng':'src', 'text_fra':'ref', 'mt_opus':'mt'})
               .to_dict(orient='records')
               )
flores_t5 = (flores_output
             .loc[:,['text_eng', 'text_fra', 'mt_t5']]
             .rename(columns={'text_eng':'src', 'text_fra':'ref', 'mt_t5':'mt'})
             .to_dict(orient='records')
             )
wmt14_output = pd.read_parquet('data/test_set/wmt14_mt_output.parquet')
wmt_opus = (wmt14_output
            .loc[:,['en', 'fr', 'mt_opus']]
            .rename(columns={'en':'src', 'fr':'ref', 'mt_opus':'mt'})
            .to_dict(orient='records')
            )
wmt_t5 = (wmt14_output
          .loc[:,['en', 'fr', 'mt_t5']]
          .rename(columns={'en':'src', 'fr':'ref', 'mt_t5':'mt'})
          .to_dict(orient='records')
          )


In [4]:
systems=['wmt_t5', 'wmt_opus', 'flores_t5', 'flores_opus']
scores = {}
system_scores = {}
for mt in tqdm(systems):
    model_output = model.predict(globals()[mt], batch_size=8)
    scores[mt] = model_output.scores
    system_scores[mt] = model_output.system_score
    print(f"Finished system: {mt}")

  0%|          | 0/4 [00:00<?, ?it/s]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/Jordan/anaconda3/envs/responsible_ml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.
 25%|██▌       | 1/4 [1:17:25<3:52:16, 4645.44s/it]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Finished system: wmt_t5


/Users/Jordan/anaconda3/envs/responsible_ml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.
 50%|█████     | 2/4 [2:29:22<2:28:24, 4452.49s/it]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Finished system: wmt_opus


/Users/Jordan/anaconda3/envs/responsible_ml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.
 75%|███████▌  | 3/4 [2:53:39<51:24, 3084.59s/it]  GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Finished system: flores_t5


/Users/Jordan/anaconda3/envs/responsible_ml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.
100%|██████████| 4/4 [3:17:34<00:00, 2963.56s/it]

Finished system: flores_opus
